In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


In [0]:
INSERT INTO ecommerce_orders 
VALUES (109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO ecommerce_orders VALUES
(110,'Arjun Nair','Kochi','Mobile',1,25000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
INSERT INTO ecommerce_orders 
VALUES (112,'Neha Kapoor','Delhi','Camera',1,60000,'Shipped');


num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO ecommerce_orders 
VALUES (113,'Ramesh Gupta','Chennai','Headphones',5,2000,'Placed');


num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO ecommerce_orders VALUES
(114,'Suresh Rao','Hyderabad','Mobile',1,28000,'Placed'),
(115,'Lakshmi Devi','Hyderabad','Tablet',2,26000,'Placed'),
(116,'Venkat Reddy','Hyderabad','Laptop',1,75000,'Placed');

num_affected_rows,num_inserted_rows
3,3


In [0]:
UPDATE ecommerce_orders 
SET order_status = 'Shipped' 
WHERE order_id = 101;


num_affected_rows
1


In [0]:
UPDATE ecommerce_orders 
SET quantity = quantity + 1 
WHERE order_id = 102;

num_affected_rows
1


In [0]:
UPDATE ecommerce_orders 
SET price = 78000 
WHERE product = 'Laptop';

num_affected_rows
5


In [0]:
UPDATE ecommerce_orders 
SET city = 'Secunderabad' 
WHERE customer_name = 'Amit Sharma';


num_affected_rows
1


In [0]:
UPDATE ecommerce_orders 
SET order_status = 'Delivered' 
WHERE product = 'Mobile';


num_affected_rows
4


In [0]:
UPDATE ecommerce_orders 
SET price = 2500 
WHERE product = 'Headphones';

num_affected_rows
2


In [0]:
UPDATE ecommerce_orders 
SET price = price + 1000 
WHERE product = 'Tablet';


num_affected_rows
3


In [0]:
UPDATE ecommerce_orders 
SET order_status = 'Processing' 
WHERE city = 'Bangalore';


num_affected_rows
2


In [0]:
UPDATE ecommerce_orders 
SET quantity = 2 
WHERE quantity = 1;


num_affected_rows
11


In [0]:
UPDATE ecommerce_orders 
SET city = 'Surat' 
WHERE city = 'Ahmedabad';

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders 
WHERE order_status = 'Cancelled';

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders 
WHERE quantity > 3;


num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders 
WHERE product = 'Camera';


num_affected_rows
2


In [0]:
DELETE FROM ecommerce_orders 
WHERE city = 'Kolkata';

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders 
WHERE price < 5000;



In [0]:
DELETE FROM ecommerce_orders 
WHERE customer_name LIKE 'A%';


num_affected_rows
2


In [0]:
DELETE FROM ecommerce_orders 
WHERE product = 'Tablet';


num_affected_rows
2


In [0]:
DELETE FROM ecommerce_orders 
WHERE city = 'Mumbai' AND quantity = 1;

num_affected_rows
0


In [0]:
DELETE FROM ecommerce_orders 
WHERE price > 80000;

num_affected_rows
0


In [0]:
DELETE FROM ecommerce_orders 
WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


In [0]:
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN 
  UPDATE SET *

WHEN NOT MATCHED THEN 
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:

SELECT * FROM ecommerce_orders ORDER BY order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
103,Rohit Mehta,Mumbai,Headphones,3,2500,Shipped
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
114,Suresh Rao,Hyderabad,Mobile,2,28000,Delivered


In [0]:

SELECT 
  t.order_id,
  CASE 
    WHEN s.order_id IS NULL THEN 'Existing'
    WHEN t.order_id = s.order_id THEN 'Updated/Inserted'
  END AS status
FROM ecommerce_orders t
LEFT JOIN incoming_orders s
ON t.order_id = s.order_id;

order_id,status
108,Existing
114,Existing
103,Existing
102,Updated/Inserted
104,Updated/Inserted
109,Updated/Inserted
110,Updated/Inserted
111,Updated/Inserted


In [0]:
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN 
  UPDATE SET t.order_status = s.order_status

WHEN NOT MATCHED THEN 
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.quantity > t.quantity THEN
  UPDATE SET t.quantity = s.quantity

WHEN NOT MATCHED THEN 
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.order_status != 'Cancelled' THEN
  UPDATE SET *

WHEN NOT MATCHED AND s.order_status != 'Cancelled' THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN NOT MATCHED AND s.product = 'Laptop' THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:

MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.price > t.price THEN
  UPDATE SET t.price = s.price

WHEN NOT MATCHED THEN 
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:

MERGE INTO ecommerce_orders t
USING daily_updates s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET 
    t.quantity = s.quantity,
    t.price = s.price,
    t.order_status = s.order_status

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:

SELECT COUNT(*) AS updated_rows
FROM ecommerce_orders t
JOIN daily_updates s
ON t.order_id = s.order_id;

updated_rows
3


In [0]:

SELECT COUNT(*) AS inserted_rows
FROM daily_updates s
LEFT JOIN ecommerce_orders t
ON t.order_id = s.order_id
WHERE t.order_id IS NULL;

inserted_rows
0


In [0]:

SELECT * 
FROM ecommerce_orders 
ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
114,Suresh Rao,Hyderabad,Mobile,2,28000,Delivered
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
112,Sanjay Gupta,Delhi,Tablet,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
